## Plan de travail
- **Partie 1 :** implémentation d"une couche GAT single-head en NumPy (Exos 1.1 → 1.4 + Quiz 1.1).
- **Partie 2 :** architecture GAT PyG sur Cora, entraînement + Quiz 2.1.
- **Partie 3 :** extraction et visualisation des poids d"attention.
- **Partie 4 :** comparaison GCN / GAT / GATv2 (perf & temps).
- **Annexes :** rapport synthétique + section sur l"usage du GenAI.

In [1]:
# Chargement des dépendances principales et du dataset Cora pour PyG
import os
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures

torch.manual_seed(42)
np.random.seed(42)
DATA_ROOT = "Cora"
dataset = Planetoid(root=DATA_ROOT, name="Cora", transform=NormalizeFeatures())
data = dataset[0]
print(dataset)
print(data)

Cora()
Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])


Processing...
Done!
c:\Users\asus-\Projets\GNN\.venv\Lib\site-packages\torch_geometric\datasets\planetoid.py:94: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, sel

## Partie 1 · NumPy GAT (Exos 1.1 → 1.4)
Compléter progressivement la classe `GATLayerNumPy` ci-dessous : initialisation Xavier, LeakyReLU, coefficients d"attention masqués par l"adjacence et passage avant complet. Chaque TODO renvoie au numéro d"exercice correspondant. Pensez à documenter les réponses aux quiz directement sous les cellules concernées.

In [2]:
import math

class GATLayerNumPy:
    """Couche GAT single-head en NumPy utilisée dans la Partie 1.

    Les méthodes sont à compléter suivant les exercices décrits dans 3-GAT.md.
    """

    def __init__(self, in_features: int, out_features: int, alpha: float = 0.2):
        self.in_features = in_features
        self.out_features = out_features
        self.alpha = alpha
        limit = math.sqrt(6.0 / (in_features + out_features))
        rng = np.random.default_rng(42)
        self.W = rng.uniform(-limit, limit, size=(in_features, out_features))  # Exo 1.1
        self.a = rng.uniform(-limit, limit, size=(2 * out_features, 1))        # Exo 1.1

    @staticmethod
    def leaky_relu(x, negative_slope: float = 0.2):
        """Exo 1.2 : implémenter la variante NumPy du LeakyReLU.
        """
        raise NotImplementedError("Compléter le LeakyReLU (Exercice 1.2).")

    @staticmethod
    def masked_softmax(logits: np.ndarray, mask: np.ndarray):
        """Utilitaire pour appliquer un softmax stable en ne gardant que les voisins (mask = adj).
        """
        raise NotImplementedError("Compléter le masked softmax (Exercice 1.3).")

    def compute_attention_coefficients(self, h: np.ndarray, adj: np.ndarray):
        """Exo 1.3 : calculer e_ij, appliquer LeakyReLU, masquer avec adj, puis normaliser.
        """
        raise NotImplementedError("Compléter le calcul des coefficients d'attention (Exercice 1.3).")

    def forward(self, features: np.ndarray, adj: np.ndarray, activation=np.tanh):
        """Exo 1.4 : propager les features via l'aggrégation attentionnelle puis activer.
        """
        raise NotImplementedError("Compléter la passe avant (Exercice 1.4).")

## Partie 2 · Pipeline PyG GAT
Définir ensuite l"architecture `GAT` (2 couches GATConv), la fonction `train()` et les métriques de validation / test. Utiliser les blocs ci-dessous pour structurer l"implémentation avant de passer à la visualisation des poids (Partie 3).

> Astuce : préparez aussi une cellule dédiée aux hyperparamètres pour faciliter les comparaisons ultérieures (Partie 4).